In [ ]:
# 1

MODEL_DIR = "/content/drive/MyDrive/llama_models/llama3_2_2b"
DATA_FILE = "/content/drive/MyDrive/llm_datasets/marathi/mr_clean_500k.txt"
OUTPUT_DIR = "/content/drive/MyDrive/llama_stage1_marathi_qlora"

In [ ]:
# 2

!pip install -q \
  transformers \
  datasets \
  accelerate \
  bitsandbytes \
  peft \
  sentencepiece

**Count the number of tokens from downloaded data.**

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_DIR,
    use_fast=True
)

print("Tokenizer loaded:", tokenizer.__class__.__name__)

Tokenizer loaded: PreTrainedTokenizerFast


In [ ]:
from tqdm import tqdm

total_tokens = 0
total_lines = 0
empty_lines = 0

with open(DATA_FILE, encoding="utf-8") as f:
    for line in tqdm(f, desc="Counting tokens"):
        line = line.strip()
        if not line:
            empty_lines += 1
            continue

        tokens = tokenizer.encode(
            line,
            add_special_tokens=False
        )

        total_tokens += len(tokens)
        total_lines += 1

print("\n===== TOKEN STATS =====")
print("Total lines:", total_lines)
print("Empty lines:", empty_lines)
print("Total tokens:", total_tokens)
print("Average tokens per line:", round(total_tokens / total_lines, 2))

Counting tokens: 0it [00:00, ?it/s]


NameError: name 'tokenizer' is not defined

**Casual LM Training (Stage - 1)**

In [ ]:
# 3

!pip install -q transformers datasets accelerate bitsandbytes peft trl

In [ ]:
# 4

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_DIR,
    local_files_only=True,
    trust_remote_code=True
)

# Required for CLM batching
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
# import random

# TARGET_LINES = 400_000   # ≈ 60M tokens
# random.seed(42)

# with open(DATA_FILE, encoding="utf-8") as f:
#     all_lines = [line.strip() for line in f if line.strip()]

# selected_lines = random.sample(all_lines, TARGET_LINES)

# print("Selected lines:", len(selected_lines))

In [ ]:
# # 5

# LINES_PER_DAY = 100_000
# DAY = 1   # Day 0, Day 1, Day 2, ... # Day 0 done.

In [ ]:
# Alternate Code

# 5

NEW_LINES_PER_DAY = 70_000
NEW_DAY = 2 # New_Day 1 Done

In [ ]:
# # 6

# START = DAY * LINES_PER_DAY
# END   = (DAY + 1) * LINES_PER_DAY

# selected_lines = []

# with open(DATA_FILE, encoding="utf-8") as f:
#     for i, line in enumerate(f):
#         if i < START:
#             continue
#         if i == START or i == END-1:
#             print(i, line.strip())
#         if i >= END:
#             break
#         if line.strip():
#             selected_lines.append(line.strip())

# print(f"Day {DAY}: loaded lines {START} → {END}")
# print("Selected lines:", len(selected_lines))

100000 जो ने तडक स्वतःचा सेल हातात घेतला.. हळूच तो नंबर डायल केला.. आणि नांव दिसले..
199999 ठेवण्यासाठी चांगल्या हर्बलचा वापर करा. यासाठी पुन्हा-पुन्हा फेसवॉश लावण्या पेक्षा साध्या
Day 1: loaded lines 100000 → 200000
Selected lines: 100000


In [ ]:
# Alternate Code

# 6

NEW_START = 100_000 + (NEW_DAY * NEW_LINES_PER_DAY)
NEW_END   = 100_000 + ((NEW_DAY + 1) * NEW_LINES_PER_DAY)

selected_lines = []

with open(DATA_FILE, encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i < NEW_START:
            continue
        if i == NEW_START or i == NEW_END-1:
            print(i, line.strip())
        if i >= NEW_END:
            break
        if line.strip():
            selected_lines.append(line.strip())

print(f"Day {NEW_DAY}: loaded lines {NEW_START} → {NEW_END}")
print("Selected lines:", len(selected_lines))

170000 काल जो बायडेन यांच्या विजायवर शिक्कामोर्तब करण्यासाठी अमेरिकेच्या संसदेची बैठक बोलावण्यात आली होती. या बैठकीत जो बायडन यांच्या विजयाची अधिकृत घोषणा केली जाणार होती. परंतु त्याचवेळी ट्रम्प समर्थक कॅपिटॉल इमारतीमध्ये घुसले आणि गोंधळ घालायला सुरूवात केली. त्यामुळे संसदेचे कामकाज थांबवावले गेले. ट्रम्प समर्थकांची पोलिसांसोबत झटापटही झाली. ट्रंप समर्थक सभागृहाच्या दरवाज्यापर्यंत आले होते. पोलिसानी त्यांच्यावर पिस्तूल रोखल्याची छायाचित्रही प्रसिद्ध झाली आहेत. अनेक संसद सदस्य खुर्चीखाली लपले होते.
239999 काय रात्र होती ती! सकाळी दहा वाजेपर्यंत रात्रच चाललेली होती...
Day 1: loaded lines 170000 → 240000
Selected lines: 70000


In [ ]:
# 7

from datasets import Dataset

def tokenize_and_chunk(lines, tokenizer, block_size=2048):
    all_ids = []
    for line in lines:
        ids = tokenizer.encode(line, add_special_tokens=False)
        all_ids.extend(ids + [tokenizer.eos_token_id])

    chunks = []
    for i in range(0, len(all_ids) - block_size, block_size):
        chunks.append(all_ids[i:i + block_size])

    return Dataset.from_dict({"input_ids": chunks})

dataset = tokenize_and_chunk(selected_lines, tokenizer)
print(dataset)

Dataset({
    features: ['input_ids'],
    num_rows: 5616
})


In [ ]:
# 8

import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR,
    quantization_config=bnb_config,
    device_map="auto",
    local_files_only=True,
    trust_remote_code=True
)

In [ ]:
# 9

from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model
from peft import PeftModel, prepare_model_for_kbit_training

# 1. Prepare model for QLoRA (REQUIRED)
model = prepare_model_for_kbit_training(model)

# 2. Enable gradient checkpointing
model.gradient_checkpointing_enable()
model.config.use_cache = False

# 3. Enable input gradients (VERY IMPORTANT)
model.enable_input_require_grads()


# 4. Add LoRA adapters
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    # target_modules=[
    #     "q_proj", "k_proj", "v_proj", "o_proj",
    #     "gate_proj", "up_proj", "down_proj"
    # ],
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

# # LOAD MODEL WITH PREVIOUS RUN'S ADAPTER
# PREV_ADAPTER = "/content/drive/MyDrive/llama_stage1_marathi_qlora/checkpoint-1988"
# model = PeftModel.from_pretrained(model, PREV_ADAPTER)

# Load old weights INTO new LoRA structure
PREV_ADAPTER = "/content/drive/MyDrive/llama_stage1_marathi_qlora/checkpoint-702"
model.load_adapter(PREV_ADAPTER, adapter_name="default")
model.set_adapter("default")

# 5. Sanity check (MUST be > 0)
model.print_trainable_parameters()

trainable params: 3,407,872 || all params: 1,239,222,272 || trainable%: 0.2750


In [ ]:
# from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model

# # REQUIRED: prepare model for QLoRA
# model = prepare_model_for_kbit_training(model)

# # Enable gradient checkpointing (MODEL SIDE)
# model.gradient_checkpointing_enable()
# model.config.use_cache = False

# # REQUIRED: allow gradients to flow
# model.enable_input_require_grads()

In [ ]:
# from peft import LoraConfig, get_peft_model

# lora_config = LoraConfig(
#     r=16,                         # sweet spot for 1B
#     lora_alpha=32,
#     lora_dropout=0.05,
#     bias="none",
#     task_type="CAUSAL_LM",
#     target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]
# )

# model = get_peft_model(model, lora_config)
# model.print_trainable_parameters()

trainable params: 3,407,872 || all params: 1,239,222,272 || trainable%: 0.2750


/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:282: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [ ]:
# 10

from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,   # effective batch = 8
    num_train_epochs=1,
    learning_rate=2e-4,              # optimal for QLoRA
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    fp16=True,
    fp16_full_eval=False,      # ← ADDED THIS LINE
    logging_steps=50,
    save_steps=2000,
    save_total_limit=6,     # Saving more checkpoints
    report_to="none",
    optim="paged_adamw_8bit",
    max_grad_norm=1.0
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [ ]:
# 11

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    data_collator=data_collator
)

trainer.train()

Step,Training Loss
50,1.821600
100,1.817600
150,1.831100
200,1.830100
250,1.832200
300,1.821900
350,1.812500
400,1.798400
450,1.807300
500,1.798300


Step,Training Loss
50,1.821600
100,1.817600
150,1.831100
200,1.830100
250,1.832200
300,1.821900
350,1.812500
400,1.798400
450,1.807300
500,1.798300


TrainOutput(global_step=702, training_loss=1.8127780993100244, metrics={'train_runtime': 15537.8697, 'train_samples_per_second': 0.361, 'train_steps_per_second': 0.045, 'total_flos': 6.739141231588147e+16, 'train_loss': 1.8127780993100244, 'epoch': 1.0})